# v13 — powered confirmation of the decisive comparisons

## Why this run exists

A reviewer would observe, correctly, that every conclusion in the branch-ablation section rests on
four to six seeds. Computing the achieved power from the v12 paired differences shows how weak that
evidence is:

| comparison | observed difference | sd of the difference | p at n=5 | **power** | n for 80% |
|---|---|---|---|---|---|
| CNN only vs hybrid | −0.263 | 0.227 | 0.061 | **50%** | 8 |
| LSTM only vs hybrid | +0.190 | 0.242 | 0.153 | **27%** | 15 |
| no dense head vs hybrid | −0.102 | 0.550 | 0.698 | **6%** | 229 |

The primary comparison was a coin flip, and the dense-head comparison had a six percent chance of
detecting its own effect. Reporting "no significant difference" from a test with 6% power says
nothing about the world; it says the study was too small.

Two further comparisons carry the same defect. In the confirmation stage, `dropout_dense = 0.00` and
`activation = ELU` both needed only **six** seeds for 80% power and were run at four, which is why
both sit at p ≈ 0.06.

## What is being fixed

Failing to reject a null hypothesis is not evidence that the null is true. Where the intended claim
is that two variants perform *the same*, that claim needs an equivalence test, not a t-test. This
notebook therefore reports both:

- a **paired t-test**, which can establish that two variants differ;
- a **TOST equivalence test** against a pre-registered margin, which can establish that they do not
  differ by more than an amount we are willing to call unimportant.

Each comparison then receives one of three verdicts — *different*, *equivalent*, or *inconclusive* —
rather than the binary that produced the original overstatement.

## Pre-registration

Fixed before any model is trained, and not to be revised after seeing results.

| | |
|---|---|
| seeds per variant | **30** (80% power needs 8 and 15 for the two branch comparisons) |
| alpha | 0.05, two-sided |
| equivalence margin | **0.40 RMSE** on the capped-55 scale, about 8% of the 4.9 baseline |
| primary endpoint | CNN-only against the full hybrid |
| all other comparisons | secondary, reported uncorrected and labelled as such |

The margin is derived from the data rather than chosen for convenience: at ±0.25 the observed
CNN-only effect already exceeds the margin, so equivalence would be unprovable by construction and
the test would be theatre. At ±0.40 it is answerable either way.

## What the outcome is likely to be

Projecting the observed effects and variances to n = 30 gives t(29) = −6.35 (p < 0.00001) for
CNN-only and t(29) = +4.31 (p = 0.0002) for LSTM-only. If that holds, the finding is not "neither
branch is necessary" but something sharper: **the convolutional branch earns its place and the
recurrent branch costs accuracy**. That is a stronger and more specific claim than the paper
currently makes, and it is stated here in advance so that it cannot be presented afterwards as
though it had been predicted.

Roughly 2.5 hours on one GPU.

In [ ]:
import os, json, gc, time, random, math, warnings
os.environ['PYTHONHASHSEED']='42'; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd, h5py
from sklearn.metrics import mean_squared_error
from scipy import stats
import matplotlib; matplotlib.use('Agg')
import matplotlib.pyplot as plt
import tensorflow as tf
for _g in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(_g,True)
    except RuntimeError as _e: print('memory growth:',_e)
from tensorflow.keras import backend as K, regularizers
from tensorflow.keras.layers import (Input, Conv1D, MaxPooling1D, Dropout, Dense, LSTM,
                                     Bidirectional, Concatenate, GlobalAveragePooling1D, Rescaling)
from tensorflow.keras.models import Model
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
try: tf.config.experimental.enable_op_determinism()
except Exception as e: print('determinism:',e)
print('TF',tf.__version__,'| GPU',bool(tf.config.list_physical_devices('GPU')))

# ---------------- PRE-REGISTERED ----------------
SEED=42; N_SEEDS=30; ALPHA=0.05; MARGIN=0.40
PRIMARY='CNN only'
RUN_BRANCH=True      # Part A
RUN_FACTORS=True     # Part B: the two confirmation-stage factors at p ~ 0.06
# ------------------------------------------------
SUB=50; VAL_UNITS=[5,6]; CAP=55; EPOCHS=100; PATIENCE=15
SEEDS=[SEED+1000+i for i in range(N_SEEDS)]
print('%d seeds, margin %.2f, alpha %.2f'%(N_SEEDS,MARGIN,ALPHA))

## The configuration under test

This is the configuration selected in v12 (`pso_2`), reproduced exactly. Nothing about it is
re-tuned here: the point of this run is to measure the same comparisons more precisely, not to
search again.

In [ ]:
FINAL=dict(T=20, n_conv=2, f1=171, f_growth=1.021014565802962, kernel=7, dilation=1,
           use_pool=0, readout='avg', n_lstm=1, lstm_units=57, bidir=0,
           fusion='conv', fuse_filters=49, dropout_conv=0.20536647015555695,
           dropout_dense=0.36568564920359575, l2reg=4.080309957019978e-06,
           weight_decay=5.8670307506235855e-06, dense1=45, dense2=0,
           activation='relu', optimizer='adam', lr=0.0005334569884698839,
           lr_sched='plateau', loss='huber', huber_delta=16.334471957530912,
           scaler='minmax', batch=256, use_lstm=1)

BRANCH=[('full hybrid',   dict(FINAL)),
        ('CNN only',      dict(FINAL, use_lstm=0)),
        ('LSTM only',     dict(FINAL, n_conv=0)),
        ('no dense head', dict(FINAL, dense1=0, dense2=0))]

# the two confirmation-stage factors that needed 6 seeds and were given 4
FACTORS=[('dropout_dense = 0.00', dict(FINAL, dropout_dense=0.0)),
         ('activation = ELU',     dict(FINAL, activation='elu'))]
print('branch variants:',[n for n,_ in BRANCH])
print('factor variants:',[n for n,_ in FACTORS])

In [ ]:
def read_sub(d,step,chunk=400_000):
    n=d.shape[0]; out=[]
    for a in range(0,n,chunk):
        b=min(a+chunk,n); blk=d[a:b]; f=(-a)%step
        if f<len(blk): out.append(blk[f::step].astype(np.float32,copy=False))
    return np.concatenate(out)

with h5py.File('N-CMAPSS_DS05.h5','r') as h:
    Sd=read_sub(h['X_s_dev'],SUB); Vd=read_sub(h['X_v_dev'],SUB)
    Ad=read_sub(h['A_dev'],SUB);   Yd=read_sub(h['Y_dev'],SUB)
    St=read_sub(h['X_s_test'],SUB);Vt=read_sub(h['X_v_test'],SUB)
    At=read_sub(h['A_test'],SUB);  Yt=read_sub(h['Y_test'],SUB)
FD=np.hstack([Sd,Vd]); FT=np.hstack([St,Vt])
UD,CD,YD=Ad[:,0].astype(int),Ad[:,1].astype(int),Yd.ravel().astype(np.float32)
UT,CT,YT=At[:,0].astype(int),At[:,1].astype(int),Yt.ravel().astype(np.float32)
del Sd,Vd,Ad,Yd,St,Vt,At,Yt; gc.collect()
TR_M=~np.isin(UD,VAL_UNITS); VA_M=np.isin(UD,VAL_UNITS); NFEAT=FD.shape[1]
_tr=FD[TR_M]; _mn,_mx=_tr.min(0),_tr.max(0); _rg=np.where((_mx-_mn)>1e-8,_mx-_mn,1.0)
SCALE=(1.0/_rg).astype(np.float32); OFFSET=(-_mn/_rg).astype(np.float32); del _tr; gc.collect()

def windows(F,u,c,y,mask,T):
    uu,cc,FF,yy=u[mask],c[mask],F[mask],y[mask]
    key=uu.astype(np.int64)*100000+cc; o=np.argsort(key,kind='stable')
    FF,yy,key=FF[o],yy[o],key[o]; bnd=np.flatnonzero(np.diff(key))+1
    X,Y=[],[]
    for a,b in zip(np.r_[0,bnd],np.r_[bnd,len(key)]):
        A=FF[a:b]; L=yy[a:b]
        for e in range(T,len(A)+1): X.append(A[e-T:e]); Y.append(L[e-1])
    return np.asarray(X,np.float32),np.asarray(Y,np.float32)

T=FINAL['T']
Xtr,ytr=windows(FD,UD,CD,YD,TR_M,T); Xva,yva=windows(FD,UD,CD,YD,VA_M,T)
Xte,yte=windows(FT,UT,CT,YT,np.ones(len(YT),bool),T)
ytr=np.minimum(ytr,CAP); yva=np.minimum(yva,CAP); yte_cap=np.minimum(yte,CAP)
TRIV=float(np.sqrt(mean_squared_error(yte_cap,np.full(len(yte_cap),ytr.mean()))))
print('T=%d | train %s | val %s | test %s'%(T,format(len(ytr),','),format(len(yva),','),format(len(yte),',')))
print('trivial baseline (capped-55): %.3f'%TRIV)
print('leakage check: train units %s | val units %s'%(sorted(set(UD[TR_M])),VAL_UNITS))

In [ ]:
def build(kw,seed):
    act=kw['activation']
    reg=regularizers.l2(kw['l2reg']) if kw['l2reg']>1e-6 else None
    kk=dict(kernel_initializer=tf.keras.initializers.GlorotUniform(seed=seed),kernel_regularizer=reg)
    inp=Input(shape=(kw['T'],NFEAT)); x=Rescaling(scale=SCALE,offset=OFFSET)(inp)
    seq=(kw['fusion']=='conv'); br=[]
    if kw['n_conv']>0:
        c=x
        for i in range(kw['n_conv']):
            f=max(8,int(round(kw['f1']*(kw['f_growth']**i))))
            dil=min(int(kw['dilation'])**i,max(1,int(c.shape[1])//2))
            c=Conv1D(f,kw['kernel'],padding='same',activation=act,dilation_rate=dil,**kk)(c)
            if kw['dropout_conv']>0: c=Dropout(kw['dropout_conv'],seed=seed)(c)
            if (not seq) and kw['use_pool'] and i<2 and int(c.shape[1])>2:
                c=MaxPooling1D(2,padding='same')(c)
        br.append(c if seq else GlobalAveragePooling1D()(c))
    if kw.get('use_lstm',1):
        r=x
        for j in range(kw['n_lstm']):
            cell=LSTM(kw['lstm_units'],return_sequences=(seq or j<kw['n_lstm']-1),**kk)
            r=Bidirectional(cell)(r) if kw['bidir'] else cell(r)
        br.append(r)
    if not br: raise ValueError('no branch enabled')
    if seq:
        h=br[0] if len(br)==1 else Concatenate(axis=-1)(br)
        h=Conv1D(kw['fuse_filters'],3,padding='same',activation=act,**kk)(h)
        h=GlobalAveragePooling1D()(h)
    else:
        h=br[0] if len(br)==1 else Concatenate()(br)
    if kw['dense1']>=8:
        h=Dense(kw['dense1'],activation=act,**kk)(h)
        if kw['dropout_dense']>0: h=Dropout(kw['dropout_dense'],seed=seed)(h)
    if kw['dense2']>=16: h=Dense(kw['dense2'],activation=act,**kk)(h)
    m=Model(inp,Dense(1,activation='linear',**kk)(h))
    loss=tf.keras.losses.Huber(delta=kw['huber_delta']) if kw['loss']=='huber' else 'mse'
    m.compile(optimizer=tf.keras.optimizers.Adam(kw['lr']),loss=loss); return m

def run_one(kw,seed):
    K.clear_session(); gc.collect()
    os.environ['PYTHONHASHSEED']=str(seed); random.seed(seed); np.random.seed(seed)
    tf.keras.utils.set_random_seed(seed)
    m=build(kw,seed)
    m.fit(Xtr,ytr,validation_data=(Xva,yva),epochs=EPOCHS,batch_size=kw['batch'],verbose=0,
          callbacks=[EarlyStopping('val_loss',patience=PATIENCE,restore_best_weights=True),
                     ReduceLROnPlateau('val_loss',factor=0.5,patience=7,min_lr=1e-6)])
    p=m.predict(Xte,batch_size=8192,verbose=0).ravel()
    return float(np.sqrt(mean_squared_error(yte_cap,np.clip(p,0,CAP))))

ROWS=[]
def sweep(variants,part):
    t0=time.time()
    for name,kw in variants:
        for s in SEEDS:
            v=run_one(kw,s); ROWS.append(dict(part=part,variant=name,seed=s,capped=v))
        q=[r['capped'] for r in ROWS if r['variant']==name]
        print('  %-22s %.4f +/- %.4f   (%.0f min)'%(name,np.mean(q),np.std(q,ddof=1),(time.time()-t0)/60))
        pd.DataFrame(ROWS).to_csv('ncmapss_v13_power.csv',index=False)

t0=time.time(); print('warm-up timing, one run:')
_=run_one(FINAL,SEED); print('  %.0f s -> projected %.1f h for %d runs'
      %(time.time()-t0,(time.time()-t0)*N_SEEDS*(len(BRANCH)+len(FACTORS)*RUN_FACTORS)/3600,
        N_SEEDS*(len(BRANCH)+len(FACTORS)*RUN_FACTORS)))

## Part A — branch ablation at 30 seeds

Same four variants as v12, same configuration, same paired seeds. Only the seed count changes.

In [ ]:
if RUN_BRANCH:
    print('Part A: branch ablation, %d seeds each'%N_SEEDS)
    sweep(BRANCH,'branch')
    print('checkpoint written')

## Part B — the two confirmation-stage factors

`dropout_dense = 0.00` and `activation = ELU` both reached p ≈ 0.06 at four seeds and needed six for
80% power. They are re-run here at the same 30 seeds so that the confirmation stage is not left
resting on the same defect as the ablation.

In [ ]:
if RUN_FACTORS:
    print('Part B: confirmation-stage factors, %d seeds each'%N_SEEDS)
    sweep(FACTORS,'factor')
    print('checkpoint written')

## Analysis — paired t-test and TOST equivalence

Each comparison is against the full hybrid on identical seeds. A comparison is called *different*
when the paired t-test rejects at α, *equivalent* when TOST rejects both one-sided nulls at the
pre-registered margin, and *inconclusive* when neither does — which is the honest description of
what v12 actually established.

In [ ]:
D=pd.DataFrame(ROWS)
base=D[D.variant=='full hybrid'].sort_values('seed').capped.values

def tost(d,margin,alpha=ALPHA):
    n=len(d); se=d.std(ddof=1)/np.sqrt(n); df=n-1
    p_lower=stats.t.sf((d.mean()+margin)/se,df)
    p_upper=stats.t.cdf((d.mean()-margin)/se,df)
    return max(p_lower,p_upper)

def achieved_power(d,alpha=ALPHA):
    n=len(d); se=d.std(ddof=1)/np.sqrt(n); ncp=abs(d.mean())/se; df=n-1
    crit=stats.t.ppf(1-alpha/2,df)
    return 1-stats.nct.cdf(crit,df,ncp)+stats.nct.cdf(-crit,df,ncp)

print('='*100)
print('n = %d paired seeds | equivalence margin %.2f | alpha %.2f | primary endpoint: %s'
      %(N_SEEDS,MARGIN,ALPHA,PRIMARY))
print('='*100)
print('%-22s %9s %9s %10s %11s %11s %8s  %s'
      %('variant','mean','sd','diff','p t-test','p TOST','power','verdict'))
res=[]
print('%-22s %9.4f %9.4f'%('full hybrid',base.mean(),base.std(ddof=1)))
for name in [n for n,_ in BRANCH if n!='full hybrid']+[n for n,_ in FACTORS if RUN_FACTORS]:
    q=D[D.variant==name].sort_values('seed').capped.values
    if len(q)!=len(base): continue
    d=q-base
    t_,p_=stats.ttest_rel(q,base); pe=tost(d,MARGIN); pw=achieved_power(d)
    verdict='different' if p_<ALPHA else ('equivalent' if pe<ALPHA else 'inconclusive')
    star=' <- primary' if name==PRIMARY else ''
    print('%-22s %9.4f %9.4f %+10.4f %11.5f %11.5f %8.0f%%  %s%s'
          %(name,q.mean(),q.std(ddof=1),d.mean(),p_,pe,100*pw,verdict,star))
    ci=stats.t.interval(0.95,len(d)-1,loc=d.mean(),scale=d.std(ddof=1)/np.sqrt(len(d)))
    res.append(dict(variant=name,mean=float(q.mean()),sd=float(q.std(ddof=1)),
                    diff=float(d.mean()),diff_sd=float(d.std(ddof=1)),
                    ci_lo=float(ci[0]),ci_hi=float(ci[1]),
                    p_ttest=float(p_),p_tost=float(pe),power=float(pw),
                    verdict=verdict,n=int(len(d))))
json.dump(dict(n_seeds=N_SEEDS,margin=MARGIN,alpha=ALPHA,primary=PRIMARY,
               trivial=TRIV,full_hybrid=float(base.mean()),results=res),
          open('ncmapss_v13_summary.json','w'),indent=2)
R=pd.DataFrame(res); R.to_csv('ncmapss_v13_results.csv',index=False)
print(); print('saved ncmapss_v13_power.csv, _results.csv, _summary.json')

## Figure — forest plot

The right way to present this: each comparison as a difference with a confidence interval, against
the equivalence margin. A reader can then see at once whether an interval excludes zero (different),
sits entirely inside the margin (equivalent), or spans both (inconclusive).

In [ ]:
plt.rcParams.update({'font.size':8,'font.family':'serif'})
fig,ax=plt.subplots(figsize=(4.6,2.6),dpi=300)
y=np.arange(len(R))[::-1]
cols={'different':'#C00000','equivalent':'#0070C0','inconclusive':'#888888'}
ax.axvspan(-MARGIN,MARGIN,color='#0070C0',alpha=0.07,zorder=1,label='equivalence margin')
ax.axvline(0,color='k',lw=1,zorder=2)
for yy,(_,r) in zip(y,R.iterrows()):
    ax.plot([r.ci_lo,r.ci_hi],[yy,yy],'-',color=cols[r.verdict],lw=2,zorder=3)
    ax.plot(r['diff'],yy,'o',color=cols[r.verdict],ms=6,zorder=4)
    ax.text(r.ci_hi+0.02,yy,r.verdict,fontsize=6.5,va='center',color=cols[r.verdict])
ax.set_yticks(y); ax.set_yticklabels(R.variant,fontsize=7.5)
ax.set_xlabel('Change in test RMSE vs the full hybrid (negative = better)',fontsize=7.5)
ax.set_title('Paired differences with 95%% CI, n = %d seeds'%N_SEEDS,fontsize=8.5)
ax.grid(axis='x',alpha=0.25); ax.legend(fontsize=6,loc='lower right')
plt.tight_layout(); plt.savefig('fig_v13_forest.png',dpi=300,bbox_inches='tight')
print('wrote fig_v13_forest.png')
plt.show()